# Running an EMRI PE run from a notebook

This notebook provides example usage of of the `emridispatch` code: 
- load a YAML config
- override run parameters in Python
- call `run_from_config`

It uses the minimal `impulse_config.yaml` in this directory.

Requires the impulse extra: `pip install -e .[impulse]`

In [ ]:
from emridispatch.cli import set_env_guards

# Keep per-eval BLAS/OMP single-threaded and neutralize lisatools' bare
# breakpoint(); must run before numpy is first imported to take effect.
set_env_guards()

## Load the config

`load_config` returns a `SimpleNamespace` mirroring the YAML sections
(`injection`, `data`, `sampler`, `prior`, `reparam`, `run`, ...) with all
optional defaults merged in.

In [ ]:
from emridispatch.config import load_config

config_path = "impulse_config.yaml"
cfg = load_config(config_path)

print("backend: ", cfg.sampler.backend)
print("nsamples:", cfg.sampler.nsamples)
print("outdir:  ", cfg.run.outdir)

## Override run parameters

Anything in the YAML can be overwritten on the loaded namespace before a run. 
Here we lower `nsamples` so the notebook finishes quickly and make a
separate outdir so we don't touch an existing CLI run.

In [ ]:
cfg.sampler.nsamples = 100
cfg.run.outdir = "./notebook_run"

# Other overrides include:
# cfg.sampler.start_mode = "prior"    # truth | prior | fisher
# cfg.sampler.start_jitter = 0.1
# cfg.run.seed = 42

## Set up the run directory

The CLI call copies the config file into the outdir (so the run dir
is self-contained for `emridisp-postprocess`) and points the logfile to `<outdir>/run.log`.

In [ ]:
import os
import shutil

from emridispatch.logging_utils import setup_logging

os.makedirs(cfg.run.outdir, exist_ok=True)
shutil.copyfile(config_path, os.path.join(cfg.run.outdir, "config.yaml"))

setup_logging(
    outdir=cfg.run.outdir,
    level=getattr(cfg.logging, "level", "INFO"),
    filename=getattr(cfg.logging, "file", "run.log"),
);

## Run

The function `run_from_config` builds the sampling problem (injection, likelihood, priors,
Fisher-informed bounds) and hands it to the configured backend. Use
`resume=False` to ignore any existing checkpoint/cache in the outdir.

Note: if you're running this notebook in a CPU environment or with a small GPU,
generating the Fisher matrix will take some time...

In [ ]:
from emridispatch.pipeline import run_from_config

summary = run_from_config(cfg, resume=True)
summary

## Outputs

The run directory now contains several per-temperature `chain_N.txt` files (when using `impulse`),
a `run_summary.json` file, `prior_bounds.npz`, `run.log`, and the copied
`config.yaml`. You could then run the postprocessing with the CLI (`emridisp-postprocess`,
`emridisp-plot`) or load the chains using `emridispatch.results`.

In [ ]:
sorted(os.listdir(cfg.run.outdir))